# FastAPI na Prática — Bella Tavola 🍝

> **Como usar este caderno?**  
> Este documento é seu guia de aprendizagem. Leia cada seção, estude o código de referência
> e implemente os exercícios no seu ambiente preferido (Repl.it, GitHub Codespaces, ou localmente).
> Teste suas rotas pelo Swagger UI em `http://localhost:8000/docs`.
> Os gabaritos estão nas células colapsadas logo após cada exercício — tente resolver antes de abrir!


## Setup do Ambiente

Antes de começar, certifique-se de ter instalado:

```bash
pip install fastapi uvicorn pydantic-settings
```

Para rodar o servidor durante os exercícios:

```bash
uvicorn main:app --reload
```

A flag `--reload` faz o servidor reiniciar automaticamente a cada mudança no código.  
Você verá o resultado de cada rota em `http://localhost:8000/docs`.


## O Contexto

Você foi contratado para construir a API do **Bella Tavola**, um restaurante italiano que quer digitalizar
seu cardápio e operação. Ao longo deste caderno, você vai construir essa API do zero —
começando pelas rotas mais simples, tornando-a robusta contra erros,
e finalmente organizando-a para que possa crescer.

Ao final, você terá uma API real, estruturada como se fosse para produção.


## BLOCO 1



> **Objetivo:** Dominar roteamento, tipos de parâmetros e modelos Pydantic.
> Ao final deste bloco, você conseguirá expor dados de qualquer domínio via HTTP de forma estruturada.


### Conceitos-chave do Bloco 1

**O que é uma rota?**  
Uma rota é a combinação de um método HTTP (`GET`, `POST`, `PUT`, `DELETE`) com um caminho (`/pratos`, `/pratos/1`).
No FastAPI, você define rotas com decorators:

```python
@app.get("/pratos")      # responde a GET /pratos
@app.post("/pratos")     # responde a POST /pratos
```

**Três formas de passar dados para uma rota:**

| Tipo | Exemplo de URL | Quando usar |
|---|---|---|
| Path parameter | `/pratos/42` | Identificar um recurso específico |
| Query parameter | `/pratos?categoria=massa` | Filtrar, paginar, ordenar |
| Request body | `POST /pratos` com JSON | Enviar dados complexos |

**Modelos Pydantic:**  
São classes que descrevem a estrutura dos dados. O FastAPI os usa para validar automaticamente
o que entra e o que sai da sua API.

```python
from pydantic import BaseModel

class Prato(BaseModel):
    nome: str
    preco: float
    categoria: str
```

Quando você usa esse modelo como parâmetro de uma rota POST, o FastAPI automaticamente
lê o JSON da requisição, valida os tipos e entrega um objeto `Prato` pronto para usar.


### Exercício 1.1 — A porta de entrada

**Conceito:** Rota GET simples, retorno de dicionário.

O primeiro passo é criar o arquivo `main.py` com a instância do FastAPI e uma rota raiz que apresenta o restaurante.

**Referência:**


In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title="Bella Tavola API",
    description="API do restaurante Bella Tavola",
    version="1.0.0"
)

@app.get("/")
async def root():
    return {
        "restaurante": "Bella Tavola",
        "mensagem": "Bem-vindo à nossa API"
    }


**Sua tarefa:**  
Crie o arquivo `main.py` com a rota raiz. Adicione também os campos `chef`, `cidade` e `especialidade` à resposta.
Rode o servidor e acesse `http://localhost:8000/` para verificar o resultado.
Depois acesse `http://localhost:8000/docs` e observe a documentação gerada automaticamente.

**O que observar:** O FastAPI usa os metadados `title`, `description` e `version` para gerar a documentação.
Qualquer dicionário retornado de uma rota é automaticamente convertido em JSON.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente)
# Arquivo: main.py

<details>
<summary>💡 Gabarito</summary>

In [ ]:
# @title
from fastapi import FastAPI

app = FastAPI(
    title="Bella Tavola API",
    description="API do restaurante Bella Tavola",
    version="1.0.0"
)

@app.get("/")
async def root():
    return {
        "restaurante": "Bella Tavola",
        "mensagem": "Bem-vindo à nossa API",
        "chef": "Marco Rossi",
        "cidade": "São Paulo",
        "especialidade": "Massas artesanais"
    }


### Exercício 1.2 — O cardápio completo

**Conceito:** GET retornando uma lista, dados em memória.

Por enquanto, vamos guardar os pratos em uma lista Python. Em um sistema real, esses dados viriam
de um banco de dados — mas o comportamento da rota é idêntico.

**Referência:**


In [ ]:
pratos = [
    {"id": 1, "nome": "Margherita", "categoria": "pizza", "preco": 45.0},
    {"id": 2, "nome": "Carbonara", "categoria": "massa", "preco": 52.0},
]

@app.get("/pratos")
async def listar_pratos():
    return pratos


**Sua tarefa:**  
Adicione ao `main.py` uma lista com pelo menos 6 pratos do Bella Tavola, cobrindo pelo menos 3 categorias
(ex: pizza, massa, sobremesa). Crie a rota `GET /pratos` que retorna essa lista.
Verifique no Swagger que a resposta é uma lista de objetos JSON.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)

<details>
<summary>💡 Gabarito</summary>


In [ ]:
pratos = [
    {"id": 1, "nome": "Margherita", "categoria": "pizza", "preco": 45.0},
    {"id": 2, "nome": "Carbonara", "categoria": "massa", "preco": 52.0},
    {"id": 3, "nome": "Lasanha Bolonhesa", "categoria": "massa", "preco": 58.0},
    {"id": 4, "nome": "Tiramisù", "categoria": "sobremesa", "preco": 28.0},
    {"id": 5, "nome": "Quattro Stagioni", "categoria": "pizza", "preco": 49.0},
    {"id": 6, "nome": "Panna Cotta", "categoria": "sobremesa", "preco": 24.0},
]

@app.get("/pratos")
async def listar_pratos():
    return pratos


### Exercício 1.3 — Buscando um prato específico



**Conceito:** Path parameter, busca por ID, retorno condicional.

Quando o cliente quer ver os detalhes de um prato específico, ele precisa de uma rota
que receba o ID daquele prato.

**Referência:**


In [ ]:
@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    return {"mensagem": "Prato não encontrado"}


**Sua tarefa:**  
Adicione a rota `GET /pratos/{prato_id}`. Teste com IDs que existem e com IDs que não existem (ex: 99).
Observe o que acontece no segundo caso — anote sua observação, pois voltaremos a ela no Bloco 2.

**Para refletir:** O que há de errado com retornar `{"mensagem": "Prato não encontrado"}` com status 200?
Por que isso é um problema para quem consome a API?


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    return {"mensagem": "Prato não encontrado"}

# Problema: retornar status 200 quando o recurso não existe
# engana quem consome a API. Um cliente que verifica apenas
# o status code vai achar que a requisição foi bem-sucedida.
# Corrigiremos isso no Bloco 2 com HTTPException.


### Exercício 1.4 — Filtrando o cardápio

**Conceito:** Query parameters, filtragem de listas, parâmetros opcionais.

O cliente quer ver apenas as massas, ou apenas os pratos abaixo de R$50.
Query parameters são a forma certa de implementar filtros.

**Referência:**


In [ ]:
from typing import Optional

@app.get("/pratos")
async def listar_pratos(categoria: Optional[str] = None):
    if categoria:
        return [p for p in pratos if p["categoria"] == categoria]
    return pratos


**Sua tarefa:**  
Modifique a rota `GET /pratos` para aceitar dois query parameters opcionais:
`categoria` (string) e `preco_maximo` (float). A rota deve filtrar os resultados
considerando ambos os parâmetros simultaneamente quando fornecidos.
Teste os seguintes casos no Swagger:
- `/pratos` — retorna todos
- `/pratos?categoria=massa` — retorna só massas
- `/pratos?preco_maximo=50` — retorna pratos até R$50
- `/pratos?categoria=pizza&preco_maximo=47` — combina os dois filtros


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from typing import Optional

@app.get("/pratos")
async def listar_pratos(
    categoria: Optional[str] = None,
    preco_maximo: Optional[float] = None
):
    resultado = pratos

    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]

    if preco_maximo:
        resultado = [p for p in resultado if p["preco"] <= preco_maximo]

    return resultado


### Exercício 1.5 — Path e query juntos


**Conceito:** Combinação de path parameter com query parameter.

Um garçom precisa verificar se um prato específico está disponível em versão vegetariana.
A rota recebe o ID do prato (path) e um flag de filtro (query).

**Referência:**


In [ ]:
@app.get("/pratos/{prato_id}/detalhes")
async def detalhes_prato(prato_id: int, incluir_ingredientes: bool = False):
    for prato in pratos:
        if prato["id"] == prato_id:
            if incluir_ingredientes:
                return {**prato, "ingredientes": ["...lista..."]}
            return prato
    return {"mensagem": "Prato não encontrado"}


**Sua tarefa:**  
Crie a rota `GET /pratos/{prato_id}` com um query parameter opcional `formato`
que pode ser `"completo"` ou `"resumido"`. No formato resumido, retorne apenas `nome` e `preco`.
No formato completo (padrão), retorne todos os campos.
Adicione também um campo `disponivel: bool` aos seus pratos na lista
e um query parameter `apenas_disponiveis: bool = False` na rota `GET /pratos`.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# Adicionar 'disponivel' aos pratos na lista:
pratos = [
    {"id": 1, "nome": "Margherita", "categoria": "pizza", "preco": 45.0, "disponivel": True},
    {"id": 2, "nome": "Carbonara", "categoria": "massa", "preco": 52.0, "disponivel": True},
    {"id": 3, "nome": "Lasanha Bolonhesa", "categoria": "massa", "preco": 58.0, "disponivel": False},
    {"id": 4, "nome": "Tiramisù", "categoria": "sobremesa", "preco": 28.0, "disponivel": True},
    {"id": 5, "nome": "Quattro Stagioni", "categoria": "pizza", "preco": 49.0, "disponivel": True},
    {"id": 6, "nome": "Panna Cotta", "categoria": "sobremesa", "preco": 24.0, "disponivel": True},
]

@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int, formato: str = "completo"):
    for prato in pratos:
        if prato["id"] == prato_id:
            if formato == "resumido":
                return {"nome": prato["nome"], "preco": prato["preco"]}
            return prato
    return {"mensagem": "Prato não encontrado"}

@app.get("/pratos")
async def listar_pratos(
    categoria: Optional[str] = None,
    preco_maximo: Optional[float] = None,
    apenas_disponiveis: bool = False
):
    resultado = pratos
    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]
    if preco_maximo:
        resultado = [p for p in resultado if p["preco"] <= preco_maximo]
    if apenas_disponiveis:
        resultado = [p for p in resultado if p["disponivel"]]
    return resultado


### Exercício 1.6 — Cadastrando um prato


**Conceito:** POST com modelo Pydantic de entrada, geração de ID.

Agora o restaurante precisa adicionar novos pratos ao cardápio.
Para isso, usamos `POST` com um body JSON estruturado por um modelo Pydantic.

**Referência:**


In [ ]:
from pydantic import BaseModel

class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float
    disponivel: bool = True  # valor padrão

@app.post("/pratos")
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {"id": novo_id, **prato.model_dump()}
    pratos.append(novo_prato)
    return novo_prato


**O que observar:** `prato.model_dump()` converte o modelo Pydantic em dicionário Python.
O `**` desempacota esse dicionário dentro de outro — o mesmo que escrever cada campo manualmente, mas mais limpo.

**Sua tarefa:**  
Implemente a rota `POST /pratos` usando o modelo `PratoInput`.
Adicione também um campo `descricao: Optional[str] = None` ao modelo.
Teste pelo Swagger: cadastre um novo prato e depois verifique se ele aparece no `GET /pratos`.

**Para refletir:** O que acontece com os dados cadastrados quando você reinicia o servidor? Por que isso acontece?


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from pydantic import BaseModel
from typing import Optional

class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str] = None
    disponivel: bool = True

@app.post("/pratos")
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {"id": novo_id, **prato.model_dump()}
    pratos.append(novo_prato)
    return novo_prato

# Os dados são perdidos ao reiniciar porque estão em memória RAM.
# Uma variável Python só existe enquanto o processo está rodando.
# Persistência real exigiria um banco de dados.


### Exercício 1.7 — Separando entrada e saída


**Conceito:** Modelos de input e output distintos, `response_model`.

Há informações que você quer receber do cliente mas não quer devolver (ex: dados internos),
e informações que você quer incluir na resposta mas que o cliente não envia (ex: ID gerado).
Para isso, modelos separados de entrada e saída são a solução correta.

**Referência:**


In [ ]:
class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    criado_em: str  # campo gerado pelo servidor, não enviado pelo cliente

@app.post("/pratos", response_model=PratoOutput)
async def criar_prato(prato: PratoInput):
    from datetime import datetime
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {
        "id": novo_id,
        "criado_em": datetime.now().isoformat(),
        **prato.model_dump()
    }
    pratos.append(novo_prato)
    return novo_prato


**O que observar:** O `response_model=PratoOutput` faz duas coisas: documenta no Swagger
qual é o formato da resposta, e garante que campos não declarados no output não sejam retornados
— mesmo que existam no dicionário interno.

**Sua tarefa:**  
Refatore seu `POST /pratos` para usar `PratoInput` e `PratoOutput` separados.
O output deve incluir `id`, todos os campos do input, e um campo `criado_em` com o timestamp atual.
Verifique no Swagger que a documentação agora mostra claramente o schema de entrada e o de saída.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from datetime import datetime
from pydantic import BaseModel
from typing import Optional

class PratoInput(BaseModel):
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str] = None
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    descricao: Optional[str]
    disponivel: bool
    criado_em: str

@app.post("/pratos", response_model=PratoOutput)
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {
        "id": novo_id,
        "criado_em": datetime.now().isoformat(),
        **prato.model_dump()
    }
    pratos.append(novo_prato)
    return novo_prato


### Exercício 1.8 — Desafio: as bebidas 🍷


**Conceito:** Aplicação autônoma de todos os conceitos do bloco.

O Bella Tavola quer expandir a API para incluir também o cardápio de bebidas.
Desta vez, você vai projetar e implementar tudo sozinho — sem referência de código.

**Sua tarefa:**  
Implemente o CRUD parcial de bebidas com as seguintes rotas:

| Rota | Descrição |
|---|---|
| `GET /bebidas` | Lista todas as bebidas, com filtro opcional por `tipo` e `alcoolica: bool` |
| `GET /bebidas/{bebida_id}` | Retorna uma bebida específica |
| `POST /bebidas` | Cadastra uma nova bebida |

Requisitos dos modelos:
- `BebidaInput` deve ter: `nome`, `tipo`, `preco`, `alcoolica: bool`, `volume_ml: int`
- `BebidaOutput` deve ter todos os campos do input mais `id` e `criado_em`
- Popule uma lista inicial com pelo menos 5 bebidas

**Critério de qualidade:** Sua implementação deve ser consistente com o que foi feito para pratos
— mesma estrutura, mesma qualidade de código.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente)

<details>
<summary>💡 Gabarito</summary>


In [ ]:
from datetime import datetime
from typing import Optional
from pydantic import BaseModel

bebidas = [
    {"id": 1, "nome": "Água Mineral", "tipo": "agua", "preco": 8.0, "alcoolica": False, "volume_ml": 500, "criado_em": "2024-01-01T00:00:00"},
    {"id": 2, "nome": "Chianti Classico", "tipo": "vinho", "preco": 120.0, "alcoolica": True, "volume_ml": 750, "criado_em": "2024-01-01T00:00:00"},
    {"id": 3, "nome": "San Pellegrino", "tipo": "agua", "preco": 15.0, "alcoolica": False, "volume_ml": 750, "criado_em": "2024-01-01T00:00:00"},
    {"id": 4, "nome": "Suco de Laranja", "tipo": "suco", "preco": 18.0, "alcoolica": False, "volume_ml": 300, "criado_em": "2024-01-01T00:00:00"},
    {"id": 5, "nome": "Prosecco", "tipo": "vinho", "preco": 95.0, "alcoolica": True, "volume_ml": 750, "criado_em": "2024-01-01T00:00:00"},
]

class BebidaInput(BaseModel):
    nome: str
    tipo: str
    preco: float
    alcoolica: bool
    volume_ml: int

class BebidaOutput(BaseModel):
    id: int
    nome: str
    tipo: str
    preco: float
    alcoolica: bool
    volume_ml: int
    criado_em: str

@app.get("/bebidas")
async def listar_bebidas(
    tipo: Optional[str] = None,
    alcoolica: Optional[bool] = None
):
    resultado = bebidas
    if tipo:
        resultado = [b for b in resultado if b["tipo"] == tipo]
    if alcoolica is not None:
        resultado = [b for b in resultado if b["alcoolica"] == alcoolica]
    return resultado

@app.get("/bebidas/{bebida_id}")
async def buscar_bebida(bebida_id: int):
    for bebida in bebidas:
        if bebida["id"] == bebida_id:
            return bebida
    return {"mensagem": "Bebida não encontrada"}

@app.post("/bebidas", response_model=BebidaOutput)
async def criar_bebida(bebida: BebidaInput):
    novo_id = max(b["id"] for b in bebidas) + 1
    nova_bebida = {
        "id": novo_id,
        "criado_em": datetime.now().isoformat(),
        **bebida.model_dump()
    }
    bebidas.append(nova_bebida)
    return nova_bebida


## BLOCO 2


> **Objetivo:** Uma API que quebra silenciosamente é pior do que uma que não existe.
> Neste bloco você vai aprender a comunicar erros de forma correta, validar dados na entrada
> e criar um contrato claro do que pode dar errado em cada rota.


### Conceitos-chave do Bloco 2

**O problema do silêncio:**  
No Bloco 1, quando um prato não era encontrado, retornávamos `{"mensagem": "Prato não encontrado"}`
com status HTTP 200. Isso está errado porque o status 200 significa "tudo certo".
Quem consome a API vai assumir que a requisição funcionou.

**HTTPException:**  
O FastAPI oferece `HTTPException` para retornar erros HTTP de forma correta:

```python
from fastapi import HTTPException

raise HTTPException(status_code=404, detail="Prato não encontrado")
```

**Validação com Field:**  
O Pydantic permite adicionar restrições aos campos dos modelos:

```python
from pydantic import BaseModel, Field

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    preco: float = Field(gt=0, description="Preço em reais")
    categoria: str = Field(pattern="^(pizza|massa|sobremesa)$")
```

**Os status codes mais importantes:**

| Código | Significado | Quando usar |
|---|---|---|
| 200 | OK | Requisição bem-sucedida |
| 201 | Created | Recurso criado com sucesso |
| 400 | Bad Request | Dados enviados estão errados |
| 404 | Not Found | Recurso não existe |
| 422 | Unprocessable Entity | Falha na validação dos dados |
| 500 | Internal Server Error | Erro inesperado no servidor |


### Exercício 2.1 — Identificando o problema

**Conceito:** Observação crítica, diagnóstico de API mal projetada.

Antes de corrigir, é preciso enxergar o problema com clareza.

**Sua tarefa:**  
Sem alterar nenhum código, teste os seguintes cenários na sua API atual
e anote o status code e o body de cada resposta:

1. `GET /pratos/99` — ID que não existe
2. `POST /pratos` com body `{"nome": "", "categoria": "pizza", "preco": -10}`
3. `POST /pratos` com body `{"nome": "Margherita"}` — campos obrigatórios faltando
4. `GET /pratos?preco_maximo=abc` — tipo errado no query parameter

Para cada caso, responda: **o status code retornado é o correto?** Se não, qual deveria ser?


In [ ]:
# ✏️ Use este espaço para anotar suas observações:

# Caso 1 — GET /pratos/99
# Status retornado:
# Status correto:
# Problema:

# Caso 2 — POST com dados inválidos
# Status retornado:
# Status correto:
# Problema:

# Caso 3 — POST com campos faltando
# Status retornado:
# Status correto:
# Problema:

# Caso 4 — GET com tipo errado
# Status retornado:
# Status correto:
# Problema:


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# Caso 1: GET /pratos/99
#   Status retornado: 200
#   Status correto: 404
#   Problema: A API finge que a requisição funcionou

# Caso 2: POST com nome vazio e preço negativo
#   Status retornado: 200 (o prato é cadastrado mesmo com dados inválidos)
#   Status correto: 422
#   Problema: Não há validação de regras de negócio

# Caso 3: POST com campos faltando
#   Status retornado: 422  ← FastAPI já trata corretamente!
#   Status correto: 422
#   Correto: O Pydantic já valida campos obrigatórios.

# Caso 4: GET com tipo errado
#   Status retornado: 422  ← FastAPI também já trata!
#   Status correto: 422
#   Correto: O FastAPI valida tipos de query parameters automaticamente.

# Conclusão: O FastAPI já valida estrutura e tipos.
# O que falta é validar regras de negócio e retornar 404 corretamente.


### Exercício 2.2 — Corrigindo o 404


**Conceito:** HTTPException, status codes semânticos.

Agora vamos corrigir a rota `GET /pratos/{prato_id}` para comunicar corretamente
quando um prato não existe.

**Referência:**


In [ ]:
from fastapi import FastAPI, HTTPException

@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    raise HTTPException(
        status_code=404,
        detail=f"Prato com id {prato_id} não encontrado"
    )


**Sua tarefa:**  
Atualize todas as rotas do Bloco 1 que retornam `{"mensagem": "... não encontrado"}`
para usar `HTTPException` com o status code correto. Isso inclui pratos e bebidas.
Teste no Swagger e confirme que o status retornado mudou para 404.

**Para refletir:** Por que usamos `raise` e não `return` para o HTTPException?


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from fastapi import FastAPI, HTTPException

@app.get("/pratos/{prato_id}")
async def buscar_prato(prato_id: int, formato: str = "completo"):
    for prato in pratos:
        if prato["id"] == prato_id:
            if formato == "resumido":
                return {"nome": prato["nome"], "preco": prato["preco"]}
            return prato
    raise HTTPException(
        status_code=404,
        detail=f"Prato com id {prato_id} não encontrado"
    )

@app.get("/bebidas/{bebida_id}")
async def buscar_bebida(bebida_id: int):
    for bebida in bebidas:
        if bebida["id"] == bebida_id:
            return bebida
    raise HTTPException(
        status_code=404,
        detail=f"Bebida com id {bebida_id} não encontrada"
    )

# raise é usado porque HTTPException é uma exceção Python.
# O raise interrompe imediatamente a execução da função
# e sobe a exceção para o FastAPI tratar.
# return continuaria a execução — o que não faz sentido num erro.


### Exercício 2.3 — Validando com Field


**Conceito:** `Field`, constraints de validação, mensagens de erro automáticas.

Preços negativos e nomes em branco não deveriam nem chegar à lógica da aplicação.
O lugar certo para barrar esses dados é no modelo Pydantic.

**Referência:**


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100, description="Nome do prato")
    categoria: str = Field(description="Categoria do prato")
    preco: float = Field(gt=0, description="Preço em reais, deve ser positivo")
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True


**Os principais constraints do Field:**

| Constraint | Tipo | Descrição |
|---|---|---|
| `gt` | número | maior que (greater than) |
| `ge` | número | maior ou igual (greater or equal) |
| `lt` | número | menor que |
| `le` | número | menor ou igual |
| `min_length` | string | comprimento mínimo |
| `max_length` | string | comprimento máximo |
| `pattern` | string | expressão regular |

**Sua tarefa:**  
Atualize `PratoInput` e `BebidaInput` com as seguintes validações:
- `nome`: entre 3 e 100 caracteres
- `preco`: maior que 0
- `categoria` dos pratos: deve ser uma das opções válidas (`pizza`, `massa`, `sobremesa`, `entrada`, `salada`)
- `tipo` das bebidas: deve ser uma das opções válidas (`vinho`, `agua`, `refrigerante`, `suco`, `cerveja`)
- `volume_ml` das bebidas: entre 50 e 2000

Teste enviando dados inválidos e observe a estrutura do erro 422 retornado.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)

<details>
<summary>💡 Gabarito</summary>


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

class BebidaInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    tipo: str = Field(pattern="^(vinho|agua|refrigerante|suco|cerveja)$")
    preco: float = Field(gt=0)
    alcoolica: bool
    volume_ml: int = Field(ge=50, le=2000)


### Exercício 2.4 — Regras de negócio com validadores


**Conceito:** `@field_validator`, validação além de tipos e constraints simples.

Algumas regras não podem ser expressas com `Field`. Por exemplo:
"o preço com desconto não pode ser maior que o preço original".
Para isso, o Pydantic oferece validadores customizados.

**Referência:**


In [ ]:
from pydantic import BaseModel, Field, field_validator

class PratoInput(BaseModel):
    nome: str = Field(min_length=3)
    preco: float = Field(gt=0)
    preco_promocional: float = Field(gt=0)

    @field_validator("preco_promocional")
    @classmethod
    def preco_promocional_menor_que_preco(cls, v, info):
        if "preco" in info.data and v >= info.data["preco"]:
            raise ValueError("Preço promocional deve ser menor que o preço original")
        return v


**Sua tarefa:**  
Adicione um campo opcional `preco_promocional: Optional[float] = None` ao `PratoInput`.
Crie um validador que garanta:
1. Se `preco_promocional` for fornecido, deve ser menor que `preco`
2. O desconto não pode ser maior que 50% do preço original

Teste com valores que violam cada uma das regras.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    preco_promocional: Optional[float] = Field(default=None, gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

    @field_validator("preco_promocional")
    @classmethod
    def validar_preco_promocional(cls, v, info):
        if v is None:
            return v
        if "preco" not in info.data:
            return v

        preco_original = info.data["preco"]

        if v >= preco_original:
            raise ValueError("Preço promocional deve ser menor que o preço original")

        desconto = (preco_original - v) / preco_original
        if desconto > 0.5:
            raise ValueError("Desconto não pode ser maior que 50% do preço original")

        return v


### Exercício 2.5 — Múltiplos erros em uma rota


**Conceito:** Lógica de negócio com múltiplos pontos de falha, status codes distintos.

Uma rota mais complexa pode falhar por várias razões diferentes.
Cada razão merece seu próprio status code e mensagem.

**Referência:**


In [ ]:
@app.post("/pratos/{prato_id}/aplicar_desconto")
async def aplicar_desconto(prato_id: int, percentual: float):
    # Erro 404: recurso não existe
    prato = next((p for p in pratos if p["id"] == prato_id), None)
    if not prato:
        raise HTTPException(status_code=404, detail="Prato não encontrado")

    # Erro 400: dado válido estruturalmente, mas inválido para o negócio
    if percentual <= 0 or percentual > 50:
        raise HTTPException(
            status_code=400,
            detail="Percentual de desconto deve estar entre 1% e 50%"
        )

    # Erro 400: estado atual impede a operação
    if not prato["disponivel"]:
        raise HTTPException(
            status_code=400,
            detail="Não é possível aplicar desconto em prato indisponível"
        )

    prato["preco"] = prato["preco"] * (1 - percentual / 100)
    return prato


**Sua tarefa:**  
Crie a rota `PUT /pratos/{prato_id}/disponibilidade` que altera a disponibilidade de um prato.
Ela deve falhar com 404 se o prato não existir.

Além disso, crie a rota `POST /pedidos` com o seguinte body:
```json
{"prato_id": 1, "quantidade": 2, "observacao": "sem cebola"}
```
Essa rota deve falhar com 404 se o prato não existir,
e com 400 se o prato estiver indisponível.
Se tudo estiver ok, retorne o pedido com o valor total calculado.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class DisponibilidadeInput(BaseModel):
    disponivel: bool

@app.put("/pratos/{prato_id}/disponibilidade")
async def alterar_disponibilidade(prato_id: int, body: DisponibilidadeInput):
    for prato in pratos:
        if prato["id"] == prato_id:
            prato["disponivel"] = body.disponivel
            return prato
    raise HTTPException(status_code=404, detail="Prato não encontrado")


pedidos = []

class PedidoInput(BaseModel):
    prato_id: int
    quantidade: int = Field(ge=1)
    observacao: Optional[str] = None

class PedidoOutput(BaseModel):
    id: int
    prato_id: int
    nome_prato: str
    quantidade: int
    valor_total: float
    observacao: Optional[str]

@app.post("/pedidos", response_model=PedidoOutput)
async def criar_pedido(pedido: PedidoInput):
    prato = next((p for p in pratos if p["id"] == pedido.prato_id), None)

    if not prato:
        raise HTTPException(status_code=404, detail="Prato não encontrado")

    if not prato["disponivel"]:
        raise HTTPException(
            status_code=400,
            detail=f"O prato '{prato['nome']}' não está disponível no momento"
        )

    novo_id = len(pedidos) + 1
    novo_pedido = {
        "id": novo_id,
        "prato_id": pedido.prato_id,
        "nome_prato": prato["nome"],
        "quantidade": pedido.quantidade,
        "valor_total": prato["preco"] * pedido.quantidade,
        "observacao": pedido.observacao
    }
    pedidos.append(novo_pedido)
    return novo_pedido


### Exercício 2.6 — Exception handler global


**Conceito:** `@app.exception_handler`, formato consistente de erros.

Atualmente, erros de validação (422) têm um formato diferente dos erros de HTTPException (404, 400).
Em uma API profissional, todos os erros devem ter o mesmo formato para facilitar o tratamento no cliente.

**Referência:**


In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "erro": "Dados inválidos",
            "detalhes": exc.errors(),
            "path": str(request.url)
        }
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "erro": exc.detail,
            "status": exc.status_code,
            "path": str(request.url)
        }
    )


**Sua tarefa:**  
Adicione os dois exception handlers ao seu `main.py`.
Defina um formato padrão de erro que inclua os campos:
`erro` (mensagem principal), `status` (código HTTP), `path` (URL que gerou o erro)
e `detalhes` (lista de problemas específicos, quando houver).
Teste que todos os tipos de erro agora retornam nesse formato unificado.


In [ ]:
# ✏️ Escreva sua solução aqui (no seu ambiente local)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
from fastapi import Request, HTTPException
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "erro": "Dados de entrada inválidos",
            "status": 422,
            "path": str(request.url),
            "detalhes": [
                {
                    "campo": " -> ".join(str(loc) for loc in e["loc"]),
                    "mensagem": e["msg"]
                }
                for e in exc.errors()
            ]
        }
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "erro": exc.detail,
            "status": exc.status_code,
            "path": str(request.url),
            "detalhes": []
        }
    )


</details>


---
### Exercício 2.7 — Desafio: API sob pressão 🔥

**Conceito:** Diagnóstico e correção de múltiplos problemas de robustez.

Abaixo está uma versão propositalmente quebrada de parte da API do Bella Tavola.
Ela tem **5 problemas de robustez**. Sua tarefa é identificar e corrigir todos eles.

**Identifique os 5 problemas antes de ver o gabarito.**


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

reservas = [
    {"id": 1, "mesa": 5, "nome": "Silva", "pessoas": 4, "ativa": True},
    {"id": 2, "mesa": 3, "nome": "Costa", "pessoas": 2, "ativa": False},
]

class ReservaInput(BaseModel):
    mesa: int
    nome: str
    pessoas: int

@app.get("/reservas/{reserva_id}")
async def buscar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            return r
    return {"erro": "não encontrada"}          # problema?

@app.post("/reservas")
async def criar_reserva(reserva: ReservaInput):
    # problema?
    nova = {"id": len(reservas) + 1, **reserva.model_dump(), "ativa": True}
    reservas.append(nova)
    return nova

@app.delete("/reservas/{reserva_id}")
async def cancelar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            r["ativa"] = False
            return {"mensagem": "cancelada"}
    # problema?

@app.get("/reservas")
async def listar_reservas(apenas_ativas: bool = False):
    if apenas_ativas:
        return [r for r in reservas if r["ativa"] == "true"]  # problema?
    return reservas


In [ ]:
# ✏️ Liste aqui os 5 problemas que você identificou:

# Problema 1:
# Problema 2:
# Problema 3:
# Problema 4:
# Problema 5:


<details>
<summary>💡 Gabarito — os 5 problemas e a correção</summary>


In [ ]:
# PROBLEMA 1: GET /reservas/{id} retorna 200 com {"erro": ...} quando não encontra
# PROBLEMA 2: POST aceita mesa=0, pessoas=0 ou negativos (sem validação com Field)
# PROBLEMA 3: DELETE não retorna nada (None implícito, status 200) quando não encontra
# PROBLEMA 4: listar_reservas compara bool com string "true" — nunca filtra corretamente
# PROBLEMA 5: POST não verifica se a mesa já está reservada

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI()

reservas = [
    {"id": 1, "mesa": 5, "nome": "Silva", "pessoas": 4, "ativa": True},
    {"id": 2, "mesa": 3, "nome": "Costa", "pessoas": 2, "ativa": False},
]

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=20)           # corrige problema 2
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=20)

@app.get("/reservas/{reserva_id}")
async def buscar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            return r
    raise HTTPException(status_code=404, detail="Reserva não encontrada")  # corrige problema 1

@app.post("/reservas")
async def criar_reserva(reserva: ReservaInput):
    mesa_ocupada = any(                        # corrige problema 5
        r["mesa"] == reserva.mesa and r["ativa"]
        for r in reservas
    )
    if mesa_ocupada:
        raise HTTPException(
            status_code=400,
            detail=f"Mesa {reserva.mesa} já está reservada"
        )
    nova = {"id": len(reservas) + 1, **reserva.model_dump(), "ativa": True}
    reservas.append(nova)
    return nova

@app.delete("/reservas/{reserva_id}")
async def cancelar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            r["ativa"] = False
            return {"mensagem": "Reserva cancelada com sucesso"}
    raise HTTPException(status_code=404, detail="Reserva não encontrada")  # corrige problema 3

@app.get("/reservas")
async def listar_reservas(apenas_ativas: bool = False):
    if apenas_ativas:
        return [r for r in reservas if r["ativa"]]  # corrige problema 4
    return reservas


## BLOCO 3

> **Objetivo:** Um único arquivo `main.py` não sobrevive quando a API cresce.
> Neste bloco você vai aprender a estruturar o código de forma que novas funcionalidades
> possam ser adicionadas sem quebrar o que já existe.


### Conceitos-chave do Bloco 3

**O problema do `main.py` monolítico:**  
Quando todas as rotas estão em um único arquivo, qualquer mudança em qualquer parte afeta o todo.
Fica difícil de testar, de revisar em equipe e de entender onde está cada coisa.

**APIRouter:**  
O FastAPI oferece `APIRouter` para organizar rotas em grupos:

```python
# routers/pratos.py
from fastapi import APIRouter

router = APIRouter()

@router.get("/")       # responde em /pratos/
@router.get("/{id}")   # responde em /pratos/{id}
```

```python
# main.py
from routers import pratos

app.include_router(pratos.router, prefix="/pratos", tags=["Pratos"])
```

**Estrutura de projeto recomendada:**

```
bella_tavola/
├── main.py
├── config.py
├── routers/
│   ├── __init__.py
│   ├── pratos.py
│   ├── bebidas.py
│   ├── pedidos.py
│   └── reservas.py
└── models/
    ├── __init__.py
    ├── prato.py
    ├── bebida.py
    ├── pedido.py
    └── reserva.py
```

**BaseSettings:**  
Para configurações que mudam entre ambientes:

```python
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = "Bella Tavola API"
    debug: bool = False
    max_mesas: int = 20

    class Config:
        env_file = ".env"

settings = Settings()
```

Os valores podem ser sobrescritos por variáveis de ambiente ou por um arquivo `.env`
— sem mudar uma linha de código.


### Exercício 3.1 — Sentindo o problema

**Conceito:** Diagnóstico de código monolítico, motivação para refatoração.

Neste ponto, seu `main.py` provavelmente tem entre 150 e 200 linhas cobrindo
pratos, bebidas, pedidos e reservas — tudo misturado.

**Sua tarefa:**  
Sem alterar nenhum código, responda por escrito às perguntas abaixo.


In [ ]:
# ✏️ Responda às perguntas abaixo como comentários:

# 1. Se você precisasse adicionar uma funcionalidade de 'promoções',
#    em qual parte do main.py você a colocaria? Por quê?


# 2. Se dois desenvolvedores trabalhassem simultaneamente — um em pratos
#    e outro em pedidos — o que aconteceria no controle de versão (Git)?


# 3. Como você encontraria rapidamente todas as rotas relacionadas a reservas?



<details>
<summary>💡 Reflexão esperada</summary>


In [ ]:
# 1. Não há um lugar óbvio. Você provavelmente adicionaria no final do arquivo,
#    mas isso é arbitrário. Em um arquivo organizado por domínio,
#    promoções teriam seu próprio arquivo routers/promocoes.py.

# 2. Ambos editariam o mesmo arquivo main.py. Isso gera conflitos de merge
#    no Git — um dos desenvolvedores precisaria resolver manualmente
#    o que o outro fez, mesmo que as mudanças fossem em partes distintas.

# 3. Você precisaria fazer uma busca por 'reserva' no arquivo
#    e navegar manualmente. Com routers separados,
#    bastaria abrir routers/reservas.py.


### Exercício 3.2 — Extraindo o primeiro router


**Conceito:** `APIRouter`, separação básica, `include_router`.

Vamos começar extraindo apenas as rotas de pratos para um arquivo separado.

**Referência:**


In [ ]:
# routers/pratos.py
from fastapi import APIRouter, HTTPException
from typing import Optional

router = APIRouter()

pratos = [...]  # a lista de pratos fica aqui

@router.get("/")
async def listar_pratos(categoria: Optional[str] = None):
    ...

@router.get("/{prato_id}")
async def buscar_prato(prato_id: int):
    ...


In [ ]:
# main.py
from fastapi import FastAPI
from routers import pratos

app = FastAPI(title="Bella Tavola API")

app.include_router(
    pratos.router,
    prefix="/pratos",
    tags=["Pratos"]
)


ModuleNotFoundError: No module named 'routers'

**Sua tarefa:**  
Crie a pasta `routers/` e o arquivo `routers/pratos.py`.
Mova todas as rotas e a lista de pratos para lá.
Atualize o `main.py` para importar e montar o router.
Verifique que o Swagger continua funcionando e que as rotas estão agrupadas sob a tag "Pratos".

**Atenção:** Python precisa de um arquivo `routers/__init__.py` (pode estar vazio)
para reconhecer a pasta como módulo.


In [ ]:
# ✏️ Implemente no seu ambiente local.
# Estrutura esperada após o exercício:
# routers/__init__.py
# routers/pratos.py
# main.py (simplificado)


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# routers/__init__.py  → arquivo vazio

# routers/pratos.py
from fastapi import APIRouter, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
from datetime import datetime

router = APIRouter()

pratos = [
    {"id": 1, "nome": "Margherita", "categoria": "pizza", "preco": 45.0, "disponivel": True, "criado_em": "2024-01-01T00:00:00"},
    {"id": 2, "nome": "Carbonara", "categoria": "massa", "preco": 52.0, "disponivel": True, "criado_em": "2024-01-01T00:00:00"},
]

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    disponivel: bool
    criado_em: str

@router.get("/")
async def listar_pratos(categoria: Optional[str] = None, apenas_disponiveis: bool = False):
    resultado = pratos
    if categoria:
        resultado = [p for p in resultado if p["categoria"] == categoria]
    if apenas_disponiveis:
        resultado = [p for p in resultado if p["disponivel"]]
    return resultado

@router.get("/{prato_id}")
async def buscar_prato(prato_id: int):
    for prato in pratos:
        if prato["id"] == prato_id:
            return prato
    raise HTTPException(status_code=404, detail="Prato não encontrado")

@router.post("/", response_model=PratoOutput)
async def criar_prato(prato: PratoInput):
    novo_id = max(p["id"] for p in pratos) + 1
    novo_prato = {"id": novo_id, "criado_em": datetime.now().isoformat(), **prato.model_dump()}
    pratos.append(novo_prato)
    return novo_prato


# main.py
from fastapi import FastAPI
from routers import pratos

app = FastAPI(title="Bella Tavola API", version="1.0.0")

app.include_router(pratos.router, prefix="/pratos", tags=["Pratos"])


### Exercício 3.3 — Separando todos os domínios


**Conceito:** Múltiplos routers, estrutura de projeto completa.

Agora repita o processo para os domínios restantes.

**Sua tarefa:**  
Crie os arquivos:
- `routers/bebidas.py` — com as rotas de bebidas
- `routers/pedidos.py` — com as rotas de pedidos
- `routers/reservas.py` — com as rotas de reservas

Monte todos os routers no `main.py`:

```python
app.include_router(pratos.router,   prefix="/pratos",   tags=["Pratos"])
app.include_router(bebidas.router,  prefix="/bebidas",  tags=["Bebidas"])
app.include_router(pedidos.router,  prefix="/pedidos",  tags=["Pedidos"])
app.include_router(reservas.router, prefix="/reservas", tags=["Reservas"])
```

Ao final, abra o Swagger em `/docs` e confirme que as rotas estão agrupadas por domínio.


In [ ]:
# ✏️ Implemente no seu ambiente local.


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# main.py completo após separação de todos os domínios
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from routers import pratos, bebidas, pedidos, reservas

app = FastAPI(
    title="Bella Tavola API",
    description="API do restaurante Bella Tavola",
    version="1.0.0"
)

@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={
            "erro": "Dados de entrada inválidos",
            "status": 422,
            "path": str(request.url),
            "detalhes": [{"campo": " -> ".join(str(loc) for loc in e["loc"]), "mensagem": e["msg"]} for e in exc.errors()]
        }
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"erro": exc.detail, "status": exc.status_code, "path": str(request.url), "detalhes": []}
    )

app.include_router(pratos.router,   prefix="/pratos",   tags=["Pratos"])
app.include_router(bebidas.router,  prefix="/bebidas",  tags=["Bebidas"])
app.include_router(pedidos.router,  prefix="/pedidos",  tags=["Pedidos"])
app.include_router(reservas.router, prefix="/reservas", tags=["Reservas"])

@app.get("/", tags=["Geral"])
async def root():
    return {"restaurante": "Bella Tavola", "versao": "1.0.0"}


### Exercício 3.4 — Separando os modelos

**Conceito:** Pasta `models/`, reutilização de modelos entre routers.

Um problema da estrutura atual: os modelos Pydantic estão definidos dentro dos routers.
Se `PedidoOutput` precisar usar `PratoOutput`, haveria importação circular.
O lugar certo para os modelos é uma pasta separada.

**Estrutura esperada:**
```
models/
├── __init__.py
├── prato.py
├── bebida.py
├── pedido.py
└── reserva.py
```

**Referência:**


In [ ]:
# models/prato.py
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    disponivel: bool
    criado_em: str


In [ ]:
# routers/pratos.py — imports atualizados
from fastapi import APIRouter, HTTPException
from models.prato import PratoInput, PratoOutput  # importa do lugar certo
from typing import Optional
from datetime import datetime

router = APIRouter()
# ... resto do código igual


**Sua tarefa:**  
Crie a pasta `models/` e mova todos os modelos Pydantic para seus respectivos arquivos.
Atualize os imports nos routers. Verifique que tudo continua funcionando.


In [ ]:
# ✏️ Implemente no seu ambiente local.


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# models/__init__.py  → vazio

# models/prato.py
from pydantic import BaseModel, Field
from typing import Optional

class PratoInput(BaseModel):
    nome: str = Field(min_length=3, max_length=100)
    categoria: str = Field(pattern="^(pizza|massa|sobremesa|entrada|salada)$")
    preco: float = Field(gt=0)
    preco_promocional: Optional[float] = Field(default=None, gt=0)
    descricao: Optional[str] = Field(default=None, max_length=500)
    disponivel: bool = True

class PratoOutput(BaseModel):
    id: int
    nome: str
    categoria: str
    preco: float
    preco_promocional: Optional[float]
    descricao: Optional[str]
    disponivel: bool
    criado_em: str

# Os demais arquivos (bebida.py, pedido.py, reserva.py)
# seguem a mesma estrutura, com os modelos de cada domínio.


### Exercício 3.5 — Configurações com BaseSettings


**Conceito:** `BaseSettings`, variáveis de ambiente, arquivo `.env`.

Valores como o nome do restaurante, versão da API e limites de negócio não deveriam
estar hardcoded no código. Eles devem poder ser alterados por configuração de ambiente
— sem tocar no código.

```bash
pip install pydantic-settings
```

**Referência:**


In [ ]:
# config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = "Bella Tavola API"
    app_version: str = "1.0.0"
    debug: bool = False
    max_mesas: int = 20
    max_pessoas_por_mesa: int = 10

    class Config:
        env_file = ".env"

settings = Settings()


In [ ]:
# .env
# APP_NAME=Bella Tavola API
# APP_VERSION=1.0.0
# DEBUG=true
# MAX_MESAS=15
# MAX_PESSOAS_POR_MESA=8


In [ ]:
# main.py — usando settings
from config import settings

app = FastAPI(
    title=settings.app_name,
    version=settings.app_version
)


**Sua tarefa:**  
Crie o arquivo `config.py` com as configurações acima.
Crie um arquivo `.env` com pelo menos 3 configurações sobrescritas.
Use `settings` no `main.py` e nas validações onde fizer sentido
(ex: `max_mesas` para validar o campo `mesa` em reservas).
Verifique que alterar o `.env` muda o comportamento sem alterar o código.


In [ ]:
# ✏️ Implemente no seu ambiente local.


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = "Bella Tavola API"
    app_version: str = "1.0.0"
    app_description: str = "API do restaurante Bella Tavola"
    debug: bool = False
    max_mesas: int = 20
    max_pessoas_por_mesa: int = 10

    class Config:
        env_file = ".env"

settings = Settings()


# models/reserva.py — usando settings
from pydantic import BaseModel, Field
from config import settings

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=settings.max_mesas)
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=settings.max_pessoas_por_mesa)


### Exercício 3.6 — Desafio final: reservas antecipadas 🏆


**Conceito:** Implementação completa e autônoma de uma funcionalidade nova.

O Bella Tavola quer lançar um sistema de reservas antecipadas com horário.
Você vai projetar e implementar essa funcionalidade do zero, seguindo a estrutura que construímos.

**Rotas esperadas:**

| Método | Rota | Descrição |
|---|---|---|
| POST | `/reservas` | Cria uma reserva com mesa, nome, data/hora e número de pessoas |
| GET | `/reservas` | Lista reservas, com filtro por data e status |
| GET | `/reservas/{id}` | Detalha uma reserva específica |
| DELETE | `/reservas/{id}` | Cancela uma reserva |
| GET | `/reservas/mesa/{numero}` | Retorna todas as reservas de uma mesa |

**Regras de negócio:**
1. Não pode haver duas reservas ativas para a mesma mesa no mesmo dia
2. Reservas só podem ser feitas com pelo menos 1 hora de antecedência
3. O número de pessoas deve respeitar `settings.max_pessoas_por_mesa`
4. Reservas canceladas não aparecem por padrão em `GET /reservas`

**Critério de qualidade:**
- Modelos em `models/reserva.py`
- Rotas em `routers/reservas.py`
- Montado no `main.py` com prefixo e tag corretos
- HTTPException corretos em todos os pontos de falha
- Validações com `Field` e `@field_validator` nos modelos


In [ ]:
# ✏️ Implemente no seu ambiente local.
# Este é o exercício mais completo do caderno.
# Tente implementar sem olhar o gabarito primeiro.


<details>
<summary>💡 Gabarito</summary>


In [ ]:
# models/reserva.py
from pydantic import BaseModel, Field, field_validator
from typing import Optional
from datetime import datetime

class ReservaInput(BaseModel):
    mesa: int = Field(ge=1, le=20)
    nome: str = Field(min_length=2, max_length=100)
    pessoas: int = Field(ge=1, le=10)
    data_hora: datetime

    @field_validator("data_hora")
    @classmethod
    def deve_ser_futura(cls, v):
        agora = datetime.now(tz=v.tzinfo)
        if (v - agora).total_seconds() < 3600:
            raise ValueError("Reserva deve ser feita com pelo menos 1 hora de antecedência")
        return v

class ReservaOutput(BaseModel):
    id: int
    mesa: int
    nome: str
    pessoas: int
    data_hora: str
    ativa: bool
    criada_em: str


In [ ]:
# routers/reservas.py
from fastapi import APIRouter, HTTPException
from models.reserva import ReservaInput, ReservaOutput
from datetime import datetime
from typing import Optional

router = APIRouter()
reservas = []

@router.post("/", response_model=ReservaOutput)
async def criar_reserva(reserva: ReservaInput):
    data_reserva = reserva.data_hora.date()
    conflito = any(
        r["mesa"] == reserva.mesa
        and r["ativa"]
        and datetime.fromisoformat(r["data_hora"]).date() == data_reserva
        for r in reservas
    )
    if conflito:
        raise HTTPException(
            status_code=400,
            detail=f"Mesa {reserva.mesa} já está reservada para {data_reserva}"
        )
    nova = {
        "id": len(reservas) + 1,
        "mesa": reserva.mesa,
        "nome": reserva.nome,
        "pessoas": reserva.pessoas,
        "data_hora": reserva.data_hora.isoformat(),
        "ativa": True,
        "criada_em": datetime.now().isoformat()
    }
    reservas.append(nova)
    return nova

@router.get("/")
async def listar_reservas(data: Optional[str] = None, apenas_ativas: bool = True):
    resultado = reservas
    if apenas_ativas:
        resultado = [r for r in resultado if r["ativa"]]
    if data:
        resultado = [
            r for r in resultado
            if datetime.fromisoformat(r["data_hora"]).date().isoformat() == data
        ]
    return resultado

@router.get("/mesa/{numero}")
async def reservas_por_mesa(numero: int):
    return [r for r in reservas if r["mesa"] == numero]

@router.get("/{reserva_id}", response_model=ReservaOutput)
async def buscar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            return r
    raise HTTPException(status_code=404, detail="Reserva não encontrada")

@router.delete("/{reserva_id}")
async def cancelar_reserva(reserva_id: int):
    for r in reservas:
        if r["id"] == reserva_id:
            if not r["ativa"]:
                raise HTTPException(status_code=400, detail="Reserva já está cancelada")
            r["ativa"] = False
            return {"mensagem": "Reserva cancelada com sucesso"}
    raise HTTPException(status_code=404, detail="Reserva não encontrada")


# Mapa da API Construída

Ao final deste caderno, o Bella Tavola tem a seguinte API:

| Método | Rota | Descrição |
|---|---|---|
| GET | `/` | Informações do restaurante |
| GET | `/pratos` | Lista pratos com filtros |
| GET | `/pratos/{id}` | Detalha um prato |
| POST | `/pratos` | Cadastra um prato |
| PUT | `/pratos/{id}/disponibilidade` | Altera disponibilidade |
| GET | `/bebidas` | Lista bebidas com filtros |
| GET | `/bebidas/{id}` | Detalha uma bebida |
| POST | `/bebidas` | Cadastra uma bebida |
| POST | `/pedidos` | Cria um pedido |
| GET | `/reservas` | Lista reservas |
| GET | `/reservas/{id}` | Detalha uma reserva |
| GET | `/reservas/mesa/{numero}` | Reservas de uma mesa |
| POST | `/reservas` | Cria uma reserva antecipada |
| DELETE | `/reservas/{id}` | Cancela uma reserva |


# Checklist de Competências

Marque mentalmente o que você já consegue fazer:

**Bloco 1 — Surface**
- [ ] Criar uma aplicação FastAPI com rotas GET e POST
- [ ] Usar path parameters e query parameters
- [ ] Definir modelos Pydantic de entrada e saída
- [ ] Usar `response_model` para documentar e filtrar a resposta

**Bloco 2 — Robustez**
- [ ] Usar `HTTPException` com o status code correto em cada situação
- [ ] Adicionar validações com `Field` nos modelos
- [ ] Criar validadores customizados com `@field_validator`
- [ ] Implementar exception handlers globais para formato consistente de erros
- [ ] Diagnosticar problemas de robustez em uma API existente

**Bloco 3 — Estrutura**
- [ ] Separar rotas em `APIRouter` por domínio
- [ ] Organizar modelos em uma pasta `models/`
- [ ] Montar múltiplos routers no `main.py` com prefixos e tags
- [ ] Usar `BaseSettings` para configurações de ambiente
- [ ] Projetar e implementar uma funcionalidade completa de forma autônoma
